In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [ ]:
import cuml
print(cuml.__version__)

from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
print("cuML imports working")

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

In [ ]:
!pip install bertopic fastopic --quiet

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import pickle
import time
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer,ENGLISH_STOP_WORDS
from bertopic import BERTopic
from fastopic import FASTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
df = pd.read_csv("yelp_reviews_clean_CA.csv")
bn_df = pd.read_csv("yelp_academic_dataset_business.csv", low_memory=False)

df = df.merge(
    bn_df[["business_id", "name"]],
    on="business_id",
    how="left"
)
print(f"Total rows: {len(df)}")
print(f"Rows with no matching name: {df['name'].isna().sum()}")
df.head()

In [ ]:
business_sizes = df.groupby("business_id").size()
print(f"Total unique businesses: {len(business_sizes)}")
print(business_sizes.describe())
print(f"\nBusinesses with 1 review: {(business_sizes == 1).sum()}")
print(f"Businesses with 10+ reviews: {(business_sizes >= 10).sum()}")

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
MIN_REVIEWS_FOR_TOPIC_MODELING = 15

review_filler_words = {"got", "went", "really", "just", "said", "definitely",
                        "came", "asked", "told", "did", "didn't", "don't"}
custom_stopwords = list(ENGLISH_STOP_WORDS.union(review_filler_words))

def safe_filename(s):
    return re.sub(r'[\\/*?:"<>|]', "_", str(s))

def clean_for_fastopic(text, extra_stopwords):
    words = text.lower().split()
    words = [w for w in words if w not in extra_stopwords]
    return " ".join(words)

def compute_all_metrics(topic_words_list, tokenized_docs, top_n_diversity=10):
    if len(topic_words_list) == 0:
        return {
            "coherence_cv": None,
            "coherence_npmi": None,
            "coherence_umass": None,
            "diversity": None,
        }
    dictionary = Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(d) for d in tokenized_docs]

    metrics = {}
    for name, coherence_type in [("coherence_cv", "c_v"),
                                   ("coherence_npmi", "c_npmi"),
                                   ("coherence_umass", "u_mass")]:
        cm = CoherenceModel(
            topics=topic_words_list,
            texts=tokenized_docs,
            corpus=corpus,
            dictionary=dictionary,
            coherence=coherence_type
        )
        metrics[name] = cm.get_coherence()

    all_words = []
    for topic in topic_words_list:
        all_words.extend(topic[:top_n_diversity])
    metrics["diversity"] = len(set(all_words)) / len(all_words) if all_words else 0

    return metrics

business_groups = df.groupby("business_id")
total_businesses = business_groups.ngroups
print(f"Total businesses to process (BERTopic): {total_businesses}\n")

# BERTopic

In [ ]:
bertopic_results_dir = "bertopic_results"
os.makedirs(bertopic_results_dir, exist_ok=True)
bertopic_failed = []

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=10)

vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2))

start_all = time.time()

for biz_num, (biz_id, group) in enumerate(business_groups, start=1):
    result_path = f"{bertopic_results_dir}/{safe_filename(biz_id)}.pkl"

    if os.path.exists(result_path):
        print(f"[{biz_num}/{total_businesses}] {biz_id}: already done, skipping")
        continue

    biz_docs = group["text"].astype(str).tolist()
    biz_name = group["name"].iloc[0]

    if len(biz_docs) < MIN_REVIEWS_FOR_TOPIC_MODELING:
        print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): only {len(biz_docs)} reviews, skipping (below minimum)")
        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id, "business_name": biz_name,
                "num_reviews": len(biz_docs), "skipped": True, "reason": "insufficient_reviews"
            }, f)
        continue

    print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): {len(biz_docs)} reviews")

    try:
        biz_embeddings = embedding_model.encode(biz_docs, batch_size=96, show_progress_bar=False)

        min_topic_size = max(2, min(10, len(biz_docs) // 5))

        bertopic_model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            calculate_probabilities=False,
            verbose=False,
            min_topic_size=min_topic_size
        )
        bertopic_topics, _ = bertopic_model.fit_transform(biz_docs, biz_embeddings)

        topic_info = bertopic_model.get_topics()
        bertopic_topic_words = [[w for w, _ in words] for tid, words in topic_info.items() if tid != -1]

        topic_counts = Counter([int(t) for t in bertopic_topics])
        topic_counts.pop(-1, None)

        tokenized = [d.lower().split() for d in biz_docs]
        metrics = compute_all_metrics(bertopic_topic_words, tokenized)

        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id, "business_name": biz_name, "num_reviews": len(biz_docs),
                "topics": bertopic_topics, "topic_words": bertopic_topic_words,
                "topic_counts": dict(topic_counts), **metrics
            }, f)

    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        bertopic_failed.append((biz_id, biz_name, len(biz_docs), str(e)))
        continue

elapsed = time.time() - start_all
print(f"\nBERTopic done. {len(bertopic_failed)} failed out of {total_businesses}. Took {elapsed/60:.1f} min")

In [ ]:
bertopic_failures_df = pd.DataFrame(bertopic_failed, columns=["business_id", "business_name", "num_reviews", "error"])
bertopic_failures_df.sort_values("num_reviews").head(20)

# FASTopic

In [ ]:
fastopic_results_dir = "fastopic_results"
os.makedirs(fastopic_results_dir, exist_ok=True)

def process_one_business(biz_docs_cleaned, num_topics_guess):
    model = FASTopic(num_topics=num_topics_guess, verbose=False, device='cuda')
    topic_words, doc_topics = model.fit_transform(biz_docs_cleaned, learning_rate=0.02, epochs=20)
    return topic_words, doc_topics

fastopic_failed = []

start_all = time.time()

BATCH_SIZE = 6  # matches max_workers below — tune based on your GPU headroom
batch = []  # holds (biz_id, biz_name, biz_docs, biz_docs_cleaned, num_topics_guess) for businesses that need processing

def flush_batch(batch):
    """Process one batch concurrently, save results, log failures."""
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        future_to_biz = {
            executor.submit(process_one_business, item[3], item[4]): item
            for item in batch
        }
        for future in as_completed(future_to_biz):
            biz_id, biz_name, biz_docs, biz_docs_cleaned, num_topics_guess = future_to_biz[future]
            result_path = f"{fastopic_results_dir}/{safe_filename(biz_id)}.pkl"
            try:
                fastopic_topic_words, fastopic_doc_topics = future.result()

                fastopic_topic_words = [
                    topic.split() if isinstance(topic, str) else list(topic)
                    for topic in fastopic_topic_words
                ]

                doc_topics_array = np.array(fastopic_doc_topics)
                assigned_topics = doc_topics_array.argmax(axis=1) if doc_topics_array.ndim > 1 else doc_topics_array
                topic_counts = Counter([int(t) for t in assigned_topics])

                tokenized = [d.lower().split() for d in biz_docs]
                metrics = compute_all_metrics(fastopic_topic_words, tokenized)

                with open(result_path, "wb") as f:
                    pickle.dump({
                        "business_id": biz_id, "business_name": biz_name, "num_reviews": len(biz_docs),
                        "topic_words": fastopic_topic_words, "doc_topics": fastopic_doc_topics,
                        "assigned_topics": assigned_topics.tolist(),
                        "topic_counts": dict(topic_counts), **metrics
                    }, f)
                print(f"  done: {biz_id} ({biz_name})")

            except Exception as e:
                print(f"  FAILED: {biz_id} ({biz_name}) — {type(e).__name__}: {e}")
                fastopic_failed.append((biz_id, biz_name, len(biz_docs), str(e)))

for biz_num, (biz_id, group) in enumerate(business_groups, start=1):
    result_path = f"{fastopic_results_dir}/{safe_filename(biz_id)}.pkl"

    if os.path.exists(result_path):
        continue  # already done, skip silently (no need to print every skip when batching)

    biz_docs = group["text"].astype(str).tolist()
    biz_name = group["name"].iloc[0]

    if len(biz_docs) < MIN_REVIEWS_FOR_TOPIC_MODELING:
        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id, "business_name": biz_name,
                "num_reviews": len(biz_docs), "skipped": True, "reason": "insufficient_reviews"
            }, f)
        continue

    bertopic_path = f"{bertopic_results_dir}/{safe_filename(biz_id)}.pkl"
    if os.path.exists(bertopic_path):
        with open(bertopic_path, "rb") as f:
            bertopic_result = pickle.load(f)
        num_topics_guess = 5 if bertopic_result.get("skipped", False) else max(len(bertopic_result["topic_words"]), 2)
    else:
        num_topics_guess = 5

    biz_docs_cleaned = [clean_for_fastopic(d, custom_stopwords) for d in biz_docs]
    batch.append((biz_id, biz_name, biz_docs, biz_docs_cleaned, num_topics_guess))

    if len(batch) >= BATCH_SIZE:
        print(f"[~{biz_num}/{total_businesses}] processing batch of {len(batch)}...")
        flush_batch(batch)
        batch = []

# process any leftover partial batch at the end
if batch:
    print(f"processing final batch of {len(batch)}...")
    flush_batch(batch)

elapsed = time.time() - start_all
print(f"\nFASTopic done. {len(fastopic_failed)} failed out of {total_businesses}. Took {elapsed/60:.1f} min")

In [ ]:
fastopic_failures_df = pd.DataFrame(fastopic_failed, columns=["business_id", "business_name", "num_reviews", "error"])
fastopic_failures_df.sort_values("num_reviews").head(20)

In [ ]:
def show_topics(biz_id, top_n_words=8):
    bertopic_path = f"{bertopic_results_dir}/{biz_id}.pkl"
    fastopic_path = f"{fastopic_results_dir}/{biz_id}.pkl"

    if not os.path.exists(bertopic_path) or not os.path.exists(fastopic_path):
        print(f"No results found for {biz_id} in one or both folders.")
        return

    with open(bertopic_path, "rb") as f:
        b = pickle.load(f)
    with open(fastopic_path, "rb") as f:
        fst = pickle.load(f)

    if b.get("skipped") or fst.get("skipped"):
        print(f"{biz_id} was skipped (insufficient reviews).")
        return

    print(f"=== {b.get('business_name')} ({biz_id}) — {b.get('num_reviews')} reviews ===\n")

    print(f"BERTopic ({len(b['topic_words'])} topics):")
    for i, words in enumerate(b["topic_words"]):
        print(f"  Topic {i}: {', '.join(words[:top_n_words])}")

    print(f"\nFASTopic ({len(fst['topic_words'])} topics):")
    for i, words in enumerate(fst["topic_words"]):
        print(f"  Topic {i}: {', '.join(words[:top_n_words])}")

# Example usage — pick any business_id you know has finished processing
show_topics("nUqrF-h9S7myCcvNDecOvw")

# C_v Coherence, NPMI Coherence, U_Mass Coherence, and Topic Diversity

In [ ]:
comparison_rows = []

for fname in os.listdir(bertopic_results_dir):
    if not fname.endswith(".pkl"):
        continue  # skip .ipynb_checkpoints and any other non-result files

    biz_id = fname.replace(".pkl", "")
    bertopic_path = f"{bertopic_results_dir}/{fname}"
    fastopic_path = f"{fastopic_results_dir}/{fname}"

    if not os.path.exists(fastopic_path):
        continue

    with open(bertopic_path, "rb") as f:
        b = pickle.load(f)
    with open(fastopic_path, "rb") as f:
        fst = pickle.load(f)

    comparison_rows.append({
        "business_id": biz_id,
        "business_name": b.get("business_name"),
        "num_reviews": b.get("num_reviews"),
        "skipped": b.get("skipped", False),

        "bertopic_coherence_cv": b.get("coherence_cv"),
        "bertopic_coherence_npmi": b.get("coherence_npmi"),
        "bertopic_coherence_umass": b.get("coherence_umass"),
        "bertopic_diversity": b.get("diversity"),

        "fastopic_coherence_cv": fst.get("coherence_cv"),
        "fastopic_coherence_npmi": fst.get("coherence_npmi"),
        "fastopic_coherence_umass": fst.get("coherence_umass"),
        "fastopic_diversity": fst.get("diversity"),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

# Plotting the Graphs for Verification Matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
comparison_df_valid = comparison_df[~comparison_df["skipped"]].copy()

# Mean Coherence Comparison

In [ ]:
metrics_config = [
    ("coherence_cv", "C_V Coherence"),
    ("coherence_npmi", "NPMI Coherence"),
    ("coherence_umass", "U_Mass Coherence"),
    ("diversity", "Topic Diversity"),
]

for metric_key, title in metrics_config:
    bertopic_mean = comparison_df_valid[f"bertopic_{metric_key}"].mean()
    fastopic_mean = comparison_df_valid[f"fastopic_{metric_key}"].mean()

    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(["BERTopic", "FASTopic"], [bertopic_mean, fastopic_mean],
                   color=["#4C72B0", "#DD8452"])
    ax.set_ylabel(title)
    ax.set_title(f"Average {title}: BERTopic vs FASTopic")

    # label each bar with its value
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.4f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom")

    plt.tight_layout()
    plt.savefig(f"mean_{metric_key}_comparison.png", dpi=150)
    plt.show()

# Distribution of Coherence Scores

In [ ]:
for metric_key, title in metrics_config[:3]:  # only the 3 coherence metrics have meaningful distributions to compare this way
    fig, ax = plt.subplots(figsize=(6, 5))
    data = [comparison_df_valid[f"bertopic_{metric_key}"].dropna(),
            comparison_df_valid[f"fastopic_{metric_key}"].dropna()]
    ax.boxplot(data, tick_labels=["BERTopic", "FASTopic"])
    ax.set_ylabel(title)
    ax.set_title(f"{title} Distribution: BERTopic vs FASTopic")
    plt.tight_layout()
    plt.savefig(f"distribution_{metric_key}.png", dpi=150)
    plt.show()

# Coherence vs Business Size

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(comparison_df_valid["num_reviews"], comparison_df_valid["bertopic_coherence_cv"],
           alpha=0.4, label="BERTopic", s=15)
ax.scatter(comparison_df_valid["num_reviews"], comparison_df_valid["fastopic_coherence_cv"],
           alpha=0.4, label="FASTopic", s=15)
ax.set_xscale("log")
ax.set_xlabel("Number of Reviews (log scale)")
ax.set_ylabel("C_V Coherence")
ax.set_title("Coherence vs. Business Review Count")
ax.legend()
plt.tight_layout()
plt.savefig("coherence_vs_size.png", dpi=150)
plt.show()

# Win Rate

In [ ]:
comparison_df_valid["bertopic_wins"] = (
    comparison_df_valid["bertopic_coherence_cv"] > comparison_df_valid["fastopic_coherence_cv"]
)
win_counts = comparison_df_valid["bertopic_wins"].value_counts()

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [win_counts.get(True, 0), win_counts.get(False, 0)],
    labels=["BERTopic higher C_V", "FASTopic higher C_V"],
    autopct="%1.1f%%",
    colors=["#4C72B0", "#DD8452"]
)
ax.set_title("Which Model Produces Higher Coherence, Per Business")
plt.tight_layout()
plt.savefig("win_rate_pie.png", dpi=150)
plt.show()

# Ranked Topic Bars for a Selected Business

In [ ]:
def plot_topic_bars(biz_id, results_dir, model_name, top_n_topics=10):
    with open(f"{results_dir}/{biz_id}.pkl", "rb") as f:
        result = pickle.load(f)

    topic_words = result["topic_words"]
    topic_counts = result.get("topic_counts", {})

    if not topic_counts:
        print(f"No topic_counts found for {biz_id} — rerun this business to regenerate with counts.")
        return

    # Sort topics by frequency, descending
    sorted_topics = sorted(topic_counts.items(), key=lambda x: x[1], reverse=True)[:top_n_topics]

    labels = []
    sizes = []
    for topic_id, count in sorted_topics:
        idx = topic_id if topic_id >= 0 else 0  # guard for BERTopic's -1 indexing quirks
        words = topic_words[idx][:3] if idx < len(topic_words) else ["?"]
        labels.append(", ".join(words))
        sizes.append(count)

    fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.4)))
    ax.barh(labels[::-1], sizes[::-1], color="#55A868")  # reverse so highest is on top
    ax.set_xlabel("Number of Reviews")
    ax.set_title(f"{model_name} Topics — {result.get('business_name', biz_id)}")
    plt.tight_layout()
    plt.show()

# Example usage:
example_biz_id = comparison_df_valid.iloc[0]["business_id"]
plot_topic_bars(example_biz_id, bertopic_results_dir, "BERTopic")
plot_topic_bars(example_biz_id, fastopic_results_dir, "FASTopic")

In [ ]:
bertopic_index = comparison_df_valid[[
    "business_id", "business_name", "num_reviews"
]].copy()
bertopic_index["skipped"] = False
bertopic_index["coherence_cv"] = comparison_df_valid["bertopic_coherence_cv"]
bertopic_index["coherence_npmi"] = comparison_df_valid["bertopic_coherence_npmi"]
bertopic_index["coherence_umass"] = comparison_df_valid["bertopic_coherence_umass"]
bertopic_index["diversity"] = comparison_df_valid["bertopic_diversity"]
bertopic_index.to_csv("bertopic_index.csv", index=False)

fastopic_index = comparison_df_valid[[
    "business_id", "business_name", "num_reviews"
]].copy()
fastopic_index["skipped"] = False
fastopic_index["coherence_cv"] = comparison_df_valid["fastopic_coherence_cv"]
fastopic_index["coherence_npmi"] = comparison_df_valid["fastopic_coherence_npmi"]
fastopic_index["coherence_umass"] = comparison_df_valid["fastopic_coherence_umass"]
fastopic_index["diversity"] = comparison_df_valid["fastopic_diversity"]
fastopic_index.to_csv("fastopic_index.csv", index=False)

print(f"bertopic_index.csv: {len(bertopic_index)} businesses")
print(f"fastopic_index.csv: {len(fastopic_index)} businesses")